In [ ]:
# ==============================================================
# SUBSTACK BRAND STYLING
# ==============================================================
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#FDFBF7',
    'axes.facecolor': '#FAF6F0',
    'axes.edgecolor': '#E3DCD2',
    'axes.labelcolor': '#4A3B32',
    'text.color': '#4A3B32',
    'xtick.color': '#4A3B32',
    'ytick.color': '#4A3B32',
    'grid.color': '#E3DCD2',
    'grid.linestyle': '--',
    'grid.alpha': 0.7,
    'legend.facecolor': '#FAF6F0',
    'legend.edgecolor': '#E3DCD2',
    'font.family': 'sans-serif'
})

BRAND_COLORS = {
    'primary': '#A33327',
    'secondary': '#D47B5A',
    'tertiary': '#C68B59',
    'quaternary': '#E3ACA1',
    'highlight': '#5C2317',
    'muted': '#A49080'
}
# ==============================================================



# Modelo DSGE Intertemporal: País Dinámico vs País Social

Este cuaderno modela la interacción entre un país "Dinámico" ($D$) y uno "Social" ($S$) en un entorno dinámico e intertemporal con acumulación de capital y política fiscal.

## 1. Familias (Households)
Las familias maximizan su utilidad intertemporal:
$$\max \sum_{t=0}^{\infty} \beta^t \left( \frac{C_t^{1-\sigma}}{1-\sigma} - \chi \frac{L_t^{1+\varphi}}{1+\varphi} \right)$$

Sujeto a la restricción presupuestaria:
$$P_t^C C_t + P_t^I I_t + B_{t+1} + P_t^C T_t = W_t L_t + R_t K_t + (1+r_t) B_t$$

Y la ley de movimiento del capital:
$$K_{t+1} = (1-\delta) K_t + I_t$$

**Condiciones de Primer Orden (FOCs):**
1. **Oferta Laboral**: $\chi L_t^\varphi = C_t^{-\sigma} \frac{W_t}{P_t^C}$
2. **Ecuación de Euler (Bonos)**: $C_t^{-\sigma} = \beta \mathbb{E}_t \left[ C_{t+1}^{-\sigma} \frac{P_t^C}{P_{t+1}^C} (1+r_{t+1}) \right]$
3. **Ecuación de Euler (Capital)**: $1 = \beta \mathbb{E}_t \left[ \left( \frac{C_{t+1}}{C_t} \right)^{-\sigma} \left( \frac{R_{t+1}}{P_{t+1}^I} + 1 - \delta \right) \frac{P_t^I}{P_{t+1}^C} \right]$ (Asumiendo $P^I = P^C$ para simplificar).

## 2. Empresas (Firms)
Hay libre movilidad de capital y trabajo entre sectores dentro de un país.

### Sector Tecnológico (Transable, T)
$$Y_{T,t} = A_{T,t} K_{T,t}^\alpha L_{T,t}^{1-\alpha}$$
FOCs (Precio igual a costo marginal):
*   $R_t = P_{T,t} \alpha \frac{Y_{T,t}}{K_{T,t}}$
*   $W_t = P_{T,t} (1-\alpha) \frac{Y_{T,t}}{L_{T,t}}$

### Sector Social (No Transable, N)
$$Y_{N,t} = A_{N,t} K_{N,t}^\eta L_{N,t}^{1-\eta}$$
FOCs:
*   $R_t = P_{N,t} \eta \frac{Y_{N,t}}{K_{N,t}}$
*   $W_t = P_{N,t} (1-\eta) \frac{Y_{N,t}}{L_{N,t}}$

## 3. Gobierno y Política Fiscal
El gobierno recauda impuestos de suma fija $T_t$, consume bienes $G_t$ y emite deuda $D_t$.
$$P_t^C G_t + (1+r_t) D_t = P_t^C T_t + D_{t+1}$$

## 4. Comercio, Agregadores y Tipo de Cambio Real
La demanda final total $Q_t = C_t + I_t + G_t$ es un agregado CES de bienes transables ($T$) y no transables ($N$):
$$Q_t = \left[ \gamma^{\frac{1}{\phi}} (Q_{T,t})^{\frac{\phi-1}{\phi}} + (1-\gamma)^{\frac{1}{\phi}} (Q_{N,t})^{\frac{\phi-1}{\phi}} \right]^{\frac{\phi}{\phi-1}}$$
Índice de Precios al Consumidor (Numeraire $P^C = 1$ o definido por la canasta):
$$P_t^C = \left[ \gamma (P_{T,t}^{agg})^{1-\phi} + (1-\gamma) P_{N,t}^{1-\phi} \right]^{\frac{1}{1-\phi}}$$

Los transables son un agregado Armington de variedades domésticas y extranjeras.
El **Tipo de Cambio Real (TCR)** se define como la ratio de los índices de precios al consumidor (ajustados por moneda si aplica, aquí asumimos una unidad global o medimos en términos del transable D):
$$TCR = \frac{P_S^C}{P_D^C}$$

In [1]:
import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt



## Cálculo del Estado Estacionario (Steady State)
En el estado estacionario, las variables no cambian en el tiempo ($C_{t+1} = C_t$, $K_{t+1} = K_t$).
De la ecuación de Euler para bonos y capital obtenemos que la tasa de interés y el retorno del capital están fijos por los parámetros de preferencia y depreciación:
$$1+r = \frac{1}{\beta} \implies r = \frac{1-\beta}{\beta}$$
$$\frac{R}{P^C} = \frac{1}{\beta} - (1-\delta) = r + \delta$$

A continuación resolvemos numéricamente el sistema.

In [2]:
def solve_steady_state(params):
    beta = params['beta']
    delta = params['delta']
    alpha = params['alpha']
    eta = params['eta']
    omega = params['omega']
    phi = params['phi']
    gamma = params['gamma']
    sigma = params['sigma']
    varphi = params['varphi']
    chi = params['chi']
    A_TD = params['A_TD']
    A_TS = params['A_TS']
    A_ND = params['A_ND']
    A_NS = params['A_NS']
    G_D = params['G_D']
    G_S = params['G_S']

    # Retorno de capital de largo plazo fijo por preferencias
    r = (1 - beta) / beta
    R_D = r + delta
    R_S = r + delta

    def ss_system(x):
        (W_D, W_S, P_TD, P_ND, P_TS, P_NS, L_TD, L_ND, L_TS, L_NS, K_TD, K_ND, K_TS, K_NS) = x
        
        # Variables límite inferiores para estabilizar solucionador numérico
        W_D, W_S = max(W_D, 1e-4), max(W_S, 1e-4)
        P_TD, P_ND = max(P_TD, 1e-4), max(P_ND, 1e-4)
        P_TS, P_NS = max(P_TS, 1e-4), max(P_NS, 1e-4)
        L_TD, L_ND = max(L_TD, 1e-4), max(L_ND, 1e-4)
        L_TS, L_NS = max(L_TS, 1e-4), max(L_NS, 1e-4)
        K_TD, K_ND = max(K_TD, 1e-4), max(K_ND, 1e-4)
        K_TS, K_NS = max(K_TS, 1e-4), max(K_NS, 1e-4)

        # 1. Firmas: Demanda de Factores y Costo Marginal
        eq1 = R_D - P_TD * alpha * A_TD * K_TD**(alpha-1) * L_TD**(1-alpha)
        eq2 = W_D - P_TD * (1-alpha) * A_TD * K_TD**alpha * L_TD**(-alpha)
        eq3 = R_D - P_ND * eta * A_ND * K_ND**(eta-1) * L_ND**(1-eta)
        eq4 = W_D - P_ND * (1-eta) * A_ND * K_ND**eta * L_ND**(-eta)
        
        eq5 = R_S - P_TS * alpha * A_TS * K_TS**(alpha-1) * L_TS**(1-alpha)
        eq6 = W_S - P_TS * (1-alpha) * A_TS * K_TS**alpha * L_TS**(-alpha)
        eq7 = R_S - P_NS * eta * A_NS * K_NS**(eta-1) * L_NS**(1-eta)
        eq8 = W_S - P_NS * (1-eta) * A_NS * K_NS**eta * L_NS**(-eta)

        # 2. Producción
        Y_TD = A_TD * K_TD**alpha * L_TD**(1-alpha)
        Y_ND = A_ND * K_ND**eta * L_ND**(1-eta)
        Y_TS = A_TS * K_TS**alpha * L_TS**(1-alpha)
        Y_NS = A_NS * K_NS**eta * L_NS**(1-eta)

        # 3. Precios y Numeraire
        P_TD_agg = (omega * P_TD**(1-theta) + (1-omega) * P_TS**(1-theta))**(1/(1-theta))
        P_TS_agg = (omega * P_TS**(1-theta) + (1-omega) * P_TD**(1-theta))**(1/(1-theta))
        P_D_CPI = (gamma * P_TD_agg**(1-phi) + (1-gamma) * P_ND**(1-phi))**(1/(1-phi))
        P_S_CPI = (gamma * P_TS_agg**(1-phi) + (1-gamma) * P_NS**(1-phi))**(1/(1-phi))
        
        eq9 = P_D_CPI - 1.0  # El numeraire es el CPI de D

        # 4. Inversión de reposición y Capital/Trabajo Agregado
        I_D = delta * (K_TD + K_ND)
        I_S = delta * (K_TS + K_NS)
        L_D = L_TD + L_ND
        L_S = L_TS + L_NS

        # 5. Restricción Presupuestaria de Familias (Ingreso = Consumo + Inversión + Impuestos)
        # Asumimos que T = G (Gobierno balanceado en SS) para aislar efecto fiscal puramente vía recursos.
        C_D = (W_D * L_D + R_D * (K_TD + K_ND)) / P_D_CPI - I_D - G_D
        C_S = (W_S * L_S + R_S * (K_TS + K_NS)) / P_S_CPI - I_S - G_S
        C_D = max(C_D, 1e-4)
        C_S = max(C_S, 1e-4)

        # 6. Oferta Laboral
        eq10 = chi * L_D**varphi - C_D**(-sigma) * W_D / P_D_CPI
        eq11 = chi * L_S**varphi - C_S**(-sigma) * W_S / P_S_CPI

        # 7. Vaciado de Mercados y Comercio Armington
        Q_D = C_D + I_D + G_D
        Q_S = C_S + I_S + G_S
        
        E_TD = gamma * (P_TD_agg / P_D_CPI)**(1-phi) * Q_D * P_D_CPI
        E_TS = gamma * (P_TS_agg / P_S_CPI)**(1-phi) * Q_S * P_S_CPI
        
        E_T_DD = omega * (P_TD / P_TD_agg)**(1-theta) * E_TD
        E_T_DS = (1-omega) * (P_TD / P_TS_agg)**(1-theta) * E_TS

        E_ND = (1-gamma) * (P_ND / P_D_CPI)**(1-phi) * Q_D * P_D_CPI
        E_NS = (1-gamma) * (P_NS / P_S_CPI)**(1-phi) * Q_S * P_S_CPI
        
        eq12 = P_ND * Y_ND - E_ND
        eq13 = P_NS * Y_NS - E_NS
        eq14 = P_TD * Y_TD - (E_T_DD + E_T_DS)  # Mercado Transable de D vacía

        return [eq1, eq2, eq3, eq4, eq5, eq6, eq7, eq8, eq9, eq10, eq11, eq12, eq13, eq14]

    # Semilla inicial para fsolve
    x0 = [1.0]*14
    x_opt = fsolve(ss_system, x0)
    
    (W_D, W_S, P_TD, P_ND, P_TS, P_NS, L_TD, L_ND, L_TS, L_NS, K_TD, K_ND, K_TS, K_NS) = x_opt
    P_TD_agg = (omega * P_TD**(1-theta) + (1-omega) * P_TS**(1-theta))**(1/(1-theta))
    P_TS_agg = (omega * P_TS**(1-theta) + (1-omega) * P_TD**(1-theta))**(1/(1-theta))
    P_S_CPI = (gamma * P_TS_agg**(1-phi) + (1-gamma) * P_NS**(1-phi))**(1/(1-phi))
    
    # TCR = P_S / P_D. Como P_D = 1, TCR = P_S_CPI.
    TCR = P_S_CPI / 1.0
    
    return {
        'TCR (Real Exchange Rate)': TCR,
        'W_D': W_D, 'W_S': W_S,
        'C_D': (W_D*(L_TD+L_ND) + R_D*(K_TD+K_ND)) - delta*(K_TD+K_ND) - G_D,
        'C_S': (W_S*(L_TS+L_NS) + R_S*(K_TS+K_NS))/P_S_CPI - delta*(K_TS+K_NS) - G_S
    }


### Experimento: Impacto de la Política Fiscal en el Tipo de Cambio Real
Calcularemos el estado estacionario original y luego observaremos qué ocurre si el País Dinámico decide realizar una política fiscal expansiva estructural (sube su $G_D$ de 0.1 a 0.5).

In [3]:
params_base = {
    'beta': 0.96, 'delta': 0.10, 'alpha': 0.33, 'eta': 0.33,
    'omega': 0.7, 'phi': 0.5, 'gamma': 0.5, 'sigma': 1.0,
    'varphi': 1.0, 'chi': 1.0, 'theta': 2.0,
    'A_TD': 1.0, 'A_TS': 1.0, 'A_ND': 1.0, 'A_NS': 1.0,
    'G_D': 0.1, 'G_S': 0.1
}

ss_base = solve_steady_state(params_base)
print("---- Estado Estacionario Base ----")
for k, v in ss_base.items(): print(f"{k}: {v:.4f}")

# Choque fiscal permanente en D
params_fiscal = params_base.copy()
params_fiscal['G_D'] = 0.5

ss_fiscal = solve_steady_state(params_fiscal)
print("\n---- Estado Estacionario con Gasto Fiscal Elevado en País D ----")
for k, v in ss_fiscal.items(): print(f"{k}: {v:.4f}")

print(f"\nCambio en TCR: {(ss_fiscal['TCR (Real Exchange Rate)']/ss_base['TCR (Real Exchange Rate)'] - 1)*100:.2f}%")
print("Interpretación: Un mayor gasto de gobierno consume más recursos domésticos transables y no transables, apreciando la moneda doméstica frente a la extranjera (el país social se abarata comparativamente).")

NameError: name 'theta' is not defined